# 多模态RAG的跨模态召回

视觉原生的文档RAG只是一个切片，生产级的多模态RAG走得更远————在文本、图像、音频和视频工作流上做召回。

## 问题描述

单模态的RAG已经有了解决模式：嵌入查询、嵌入片段、召回、塞回给LLM。而多模态的RAG需要：
- 多个召回头（每个模块需要在兼容空间内的做嵌入）
- 模态间的召回结果做融合
- 跨模态引用来源的生成grounding
- 跨模态评估标准

## 基本概念

### 跨模态召回

给出模态A的查询，在模态B中检索语料。三种模式：
1. 共享嵌入空间。 CLIP 和 CLAP，一个是图文、一个是图音，在模态间嵌入向量的余弦相似度可以直接起作用。
2. 各模态编码器+翻译。 文本编码器+图像编码器+一个小的翻译模块在模态之间做映射。灵活但是增加了复杂度。
3. VLM作为编码器。 直接用VLM的隐藏状态作为召回表征。任何支持VLM支持的模态都能工作。质量更高，也更贵。

选择： 对于图文，CLIP + SigLIP。 对于图音，CLAP。 跨模态的VLM隐藏状态则有最好的质量。

### 融合策略

你检索到了10个结果，其中5张图片，3个文本片段，2个音频段。你怎么融合？
- 分数融合（最便宜）。 每个模态都有自己的召回器，每个都返回分数。然后各模态归一化之后求和。简单且有效。
- 基于注意力的融合。 将所有检索到的内容链接，然后让一个小的注意力网络打分。需要训练。
- MoE 融合。 门控网络路由到模块专门的专家。不同的查询类型路由结果不一样，一个视觉问题能中图像能够得到更高的得分。

生产环境的默认：分数融合加上一个查询领域的偏置。

### 生产grounding

LLM 需要表明每个声明引用的内容，对于多模态来说：
- 文本来源， 标准引用`[1]`
- 图像来源， `[img 3]`加上简短的标题
- 音频来源， `[audio 2 at 0:34]`

将生成器在ground感知的数据上进行训练，训练目标中的每个声明都被标记了引用源，模型自然吐出引用。


# 开始编码

教学积木：多模态 RAG 核心——**共享嵌入空间跨模态召回**、**多模态分数融合（+查询偏置）**、**轻量注意力融合示意**、**跨模态引用 grounding**。

> 说明：磁盘上原 `code.ipynb` 为空；上方概念笔记按编辑器内容一并写入，请核对是否与你本地一致。


## 1. 配置、文档条目与共享嵌入空间（CLIP 式）


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum

import torch
import torch.nn as nn
import torch.nn.functional as F


class Modality(str, Enum):
    """语料条目的模态类型。"""

    TEXT = "text"
    IMAGE = "image"
    AUDIO = "audio"


@dataclass
class TinyMultiModalRAGConfig:
    """多模态 RAG 教学配置。"""

    vocab_size: int = 256
    """玩具文本词表大小。"""

    max_text_len: int = 16
    """文本序列最大长度。"""

    image_size: int = 32
    """玩具图像边长。"""

    audio_len: int = 64
    """玩具音频帧数（一维特征序列占位）。"""

    audio_dim: int = 16
    """每帧音频特征维。"""

    dim: int = 64
    """共享嵌入空间维。"""

    temperature: float = 0.07
    """余弦相似度温度（CLIP 风格）。"""


@dataclass
class CorpusItem:
    """一条可检索语料。"""

    doc_id: str
    modality: Modality
    caption: str = ""
    """用于 grounding 显示的短标题。"""

    timestamp_sec: float | None = None
    """音频引用可选时间戳（秒）。"""


class TextEncoder(nn.Module):
    """玩具文本编码器：均值池化 token 嵌入 → 投影到共享空间。"""

    def __init__(self, cfg: TinyMultiModalRAGConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, token_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
        """
        Args:
            token_ids: ``(B, L)``。
            pad_id: padding id，池化时忽略。

        Returns:
            z: ``(B, D)`` L2 归一化嵌入。
        """
        mask = (token_ids != pad_id).float().unsqueeze(-1)
        h = self.emb(token_ids)
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        z = self.proj(pooled)
        return F.normalize(z, dim=-1)


class ImageEncoder(nn.Module):
    """玩具图像编码器：小 CNN → 全局池化 → 共享空间。"""

    def __init__(self, cfg: TinyMultiModalRAGConfig) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.GELU(),
            nn.Conv2d(32, cfg.dim, 4, 2, 1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            z: ``(B, D)`` L2 归一化嵌入。
        """
        h = self.net(images).flatten(1)
        return F.normalize(self.proj(h), dim=-1)


class AudioEncoder(nn.Module):
    """玩具音频编码器（CLAP 占位）：时序均值 → 共享空间。"""

    def __init__(self, cfg: TinyMultiModalRAGConfig) -> None:
        super().__init__()
        self.frame = nn.Linear(cfg.audio_dim, cfg.dim)
        self.proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        """
        Args:
            audio: ``(B, T, audio_dim)``。

        Returns:
            z: ``(B, D)`` L2 归一化嵌入。
        """
        h = self.frame(audio).mean(dim=1)
        return F.normalize(self.proj(h), dim=-1)


class SharedSpaceEncoders(nn.Module):
    """
    模式 1：共享嵌入空间。

    文本 / 图像 / 音频各自编码后落在同一 ``D`` 维球面上，跨模态用余弦相似度召回。
    """

    def __init__(self, cfg: TinyMultiModalRAGConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.text = TextEncoder(cfg)
        self.image = ImageEncoder(cfg)
        self.audio = AudioEncoder(cfg)

    def encode_query_text(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            token_ids: ``(B, L)``。

        Returns:
            z: ``(B, D)``。
        """
        return self.text(token_ids)

    def encode_corpus(self, modality: Modality, payload: torch.Tensor) -> torch.Tensor:
        """
        Args:
            modality: 语料模态。
            payload: 文本 ``(B,L)`` / 图像 ``(B,3,H,W)`` / 音频 ``(B,T,F)``。

        Returns:
            z: ``(B, D)``。
        """
        if modality is Modality.TEXT:
            return self.text(payload)
        if modality is Modality.IMAGE:
            return self.image(payload)
        if modality is Modality.AUDIO:
            return self.audio(payload)
        raise ValueError(f"unknown modality: {modality}")


print(f"shared-space encoders ready | dim={TinyMultiModalRAGConfig().dim}")


## 2. 跨模态召回 + 分数融合（含查询偏置）


In [ ]:
@dataclass
class Hit:
    """单条召回结果。"""

    doc_id: str
    modality: Modality
    score: float
    caption: str = ""
    timestamp_sec: float | None = None


def cosine_retrieve(
    query_z: torch.Tensor,
    corpus_z: torch.Tensor,
    items: list[CorpusItem],
    top_k: int,
    temperature: float = 0.07,
) -> list[Hit]:
    """
    在已对齐的共享空间中按余弦相似度召回（``query`` 可来自另一模态）。

    Args:
        query_z: ``(D,)`` 或 ``(1, D)`` 查询嵌入（已归一化）。
        corpus_z: ``(N, D)`` 语料嵌入（已归一化）。
        items: 长度 ``N`` 的元数据。
        top_k: 返回条数。
        temperature: 分数缩放 ``sim / τ``。

    Returns:
        hits: 按分数降序的 ``Hit`` 列表。
    """
    if query_z.ndim == 1:
        query_z = query_z.unsqueeze(0)
    if len(items) != corpus_z.size(0):
        raise ValueError("items length must match corpus_z")
    sim = (query_z @ corpus_z.t()).squeeze(0) / temperature
    k = min(top_k, int(sim.numel()))
    vals, idxs = torch.topk(sim, k=k)
    hits: list[Hit] = []
    for score, i in zip(vals.tolist(), idxs.tolist()):
        it = items[i]
        hits.append(
            Hit(
                doc_id=it.doc_id,
                modality=it.modality,
                score=float(score),
                caption=it.caption,
                timestamp_sec=it.timestamp_sec,
            )
        )
    return hits


def minmax_norm(scores: list[float]) -> list[float]:
    """
    Args:
        scores: 原始分数列表。

    Returns:
        normed: ``[0,1]`` 归一化；若全相等则全 ``0.5``。
    """
    if not scores:
        return []
    lo, hi = min(scores), max(scores)
    if hi - lo < 1e-8:
        return [0.5 for _ in scores]
    return [(s - lo) / (hi - lo) for s in scores]


def fuse_hits_by_score(
    hits_by_modality: dict[Modality, list[Hit]],
    query_bias: dict[Modality, float] | None = None,
    top_k: int = 5,
) -> list[Hit]:
    """
    分数融合：各模态内 min-max 归一化后加权求和（同 ``doc_id`` 合并）。

    生产默认：分数融合 + 查询域偏置（如视觉问题抬高 ``IMAGE``）。

    Args:
        hits_by_modality: 每模态的召回列表。
        query_bias: 模态权重，缺省为 1.0。
        top_k: 融合后保留条数。

    Returns:
        fused: 按融合分降序。
    """
    bias = query_bias or {}
    bucket: dict[str, Hit] = {}
    for mod, hits in hits_by_modality.items():
        if not hits:
            continue
        norms = minmax_norm([h.score for h in hits])
        w = float(bias.get(mod, 1.0))
        for h, ns in zip(hits, norms):
            fused_score = w * ns
            if h.doc_id in bucket:
                bucket[h.doc_id].score += fused_score
            else:
                bucket[h.doc_id] = Hit(
                    doc_id=h.doc_id,
                    modality=h.modality,
                    score=fused_score,
                    caption=h.caption,
                    timestamp_sec=h.timestamp_sec,
                )
    ranked = sorted(bucket.values(), key=lambda h: h.score, reverse=True)
    return ranked[:top_k]


class AttentionFusion(nn.Module):
    """
    轻量注意力融合示意：对候选嵌入做 query-conditioned 打分（需训练）。
    """

    def __init__(self, dim: int) -> None:
        super().__init__()
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)

    def forward(
        self,
        query_z: torch.Tensor,
        candidate_z: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            query_z: ``(D,)`` 或 ``(1, D)``。
            candidate_z: ``(M, D)`` 候选嵌入。

        Returns:
            weights: ``(M,)`` 注意力权重。
            fused: ``(D,)`` 加权融合向量。
        """
        if query_z.ndim == 1:
            query_z = query_z.unsqueeze(0)
        q = self.q_proj(query_z)
        k = self.k_proj(candidate_z)
        logits = (q @ k.t()).squeeze(0) / (candidate_z.size(-1) ** 0.5)
        weights = logits.softmax(dim=0)
        fused = (weights.unsqueeze(-1) * candidate_z).sum(dim=0)
        return weights, fused


print("retrieve + score/attention fusion ready")


## 3. 跨模态 grounding 引用格式


In [ ]:
def format_citation(hit: Hit, index: int) -> str:
    """
    按模态生成引用标记（笔记中的 grounding 约定）。

    Args:
        hit: 召回命中。
        index: 人类可读序号（从 1 起）。

    Returns:
        cite: 如 ``[1]`` / ``[img 3: cat]`` / ``[audio 2 at 0:34]``。
    """
    if hit.modality is Modality.TEXT:
        return f"[{index}]"
    if hit.modality is Modality.IMAGE:
        cap = hit.caption.strip() or hit.doc_id
        return f"[img {index}: {cap}]"
    if hit.modality is Modality.AUDIO:
        if hit.timestamp_sec is None:
            return f"[audio {index}]"
        m = int(hit.timestamp_sec) // 60
        s = int(hit.timestamp_sec) % 60
        return f"[audio {index} at {m}:{s:02d}]"
    raise ValueError(f"unknown modality: {hit.modality}")


def build_grounded_context(hits: list[Hit]) -> str:
    """
    把融合后的命中編成带引用标记的上下文，供生成器使用。

    Args:
        hits: 已排序的融合结果。

    Returns:
        context: 多行字符串，每行一条来源说明 + 引用标记。
    """
    lines: list[str] = []
    for i, h in enumerate(hits, start=1):
        cite = format_citation(h, i)
        lines.append(
            f"{cite} ({h.modality.value}) score={h.score:.3f} | {h.caption or h.doc_id}"
        )
    return "\n".join(lines)


def attach_claims_with_citations(claims: list[str], hits: list[Hit]) -> str:
    """
    教学示意：每个声明后挂一个来源引用（真实系统需在训练数据中标好 span）。

    Args:
        claims: 生成器产出的原子声明列表。
        hits: 可用引用池（循环分配示意）。

    Returns:
        answer: 带引用的回答文本。
    """
    if not hits:
        return " ".join(claims)
    parts: list[str] = []
    for i, claim in enumerate(claims):
        cite = format_citation(hits[i % len(hits)], (i % len(hits)) + 1)
        parts.append(f"{claim.rstrip('.')} {cite}.")
    return " ".join(parts)


print("grounding helpers ready")


## 4. 端到端玩具管线 + 冒烟测试


In [ ]:
@dataclass
class ToyIndex:
    """按模态分开存的嵌入索引。"""

    items: dict[Modality, list[CorpusItem]] = field(default_factory=dict)
    embeddings: dict[Modality, torch.Tensor] = field(default_factory=dict)


def build_toy_index(
    encoders: SharedSpaceEncoders,
    text_ids: torch.Tensor,
    text_items: list[CorpusItem],
    images: torch.Tensor,
    image_items: list[CorpusItem],
    audio: torch.Tensor,
    audio_items: list[CorpusItem],
) -> ToyIndex:
    """
    Args:
        encoders: 共享空间编码器。
        text_ids: ``(Nt, L)``。
        text_items: 长度 ``Nt``。
        images: ``(Ni, 3, H, W)``。
        image_items: 长度 ``Ni``。
        audio: ``(Na, T, F)``。
        audio_items: 长度 ``Na``。

    Returns:
        index: 分模态索引。
    """
    idx = ToyIndex()
    with torch.no_grad():
        idx.items[Modality.TEXT] = text_items
        idx.embeddings[Modality.TEXT] = encoders.encode_corpus(Modality.TEXT, text_ids)
        idx.items[Modality.IMAGE] = image_items
        idx.embeddings[Modality.IMAGE] = encoders.encode_corpus(Modality.IMAGE, images)
        idx.items[Modality.AUDIO] = audio_items
        idx.embeddings[Modality.AUDIO] = encoders.encode_corpus(Modality.AUDIO, audio)
    return idx


def multimodal_retrieve(
    encoders: SharedSpaceEncoders,
    index: ToyIndex,
    query_token_ids: torch.Tensor,
    top_k_per_modality: int = 3,
    query_bias: dict[Modality, float] | None = None,
    fused_top_k: int = 5,
) -> list[Hit]:
    """
    文本查询 → 各模态召回 → 分数融合。

    Args:
        encoders: 编码器。
        index: 索引。
        query_token_ids: ``(1, L)`` 或 ``(L,)``。
        top_k_per_modality: 每模态召回条数。
        query_bias: 模态偏置。
        fused_top_k: 融合后条数。

    Returns:
        fused_hits: 融合排序结果。
    """
    if query_token_ids.ndim == 1:
        query_token_ids = query_token_ids.unsqueeze(0)
    q = encoders.encode_query_text(query_token_ids)[0]
    per: dict[Modality, list[Hit]] = {}
    for mod in (Modality.TEXT, Modality.IMAGE, Modality.AUDIO):
        per[mod] = cosine_retrieve(
            q,
            index.embeddings[mod],
            index.items[mod],
            top_k=top_k_per_modality,
            temperature=encoders.cfg.temperature,
        )
    return fuse_hits_by_score(per, query_bias=query_bias, top_k=fused_top_k)


def smoke_test() -> None:
    """验证跨模态召回、融合、grounding。"""
    torch.manual_seed(0)
    cfg = TinyMultiModalRAGConfig()
    enc = SharedSpaceEncoders(cfg)

    text_ids = torch.randint(1, cfg.vocab_size, (5, cfg.max_text_len))
    text_items = [
        CorpusItem(doc_id=f"t{i}", modality=Modality.TEXT, caption=f"passage-{i}")
        for i in range(5)
    ]
    images = torch.randn(4, 3, cfg.image_size, cfg.image_size)
    image_items = [
        CorpusItem(doc_id=f"img{i}", modality=Modality.IMAGE, caption=f"photo-{i}")
        for i in range(4)
    ]
    audio = torch.randn(3, cfg.audio_len, cfg.audio_dim)
    audio_items = [
        CorpusItem(
            doc_id=f"a{i}",
            modality=Modality.AUDIO,
            caption=f"clip-{i}",
            timestamp_sec=12.0 + 10 * i,
        )
        for i in range(3)
    ]
    index = build_toy_index(enc, text_ids, text_items, images, image_items, audio, audio_items)

    print("=== cross-modal retrieve (text → image) ===")
    q = torch.randint(1, cfg.vocab_size, (1, cfg.max_text_len))
    qz = enc.encode_query_text(q)[0]
    img_hits = cosine_retrieve(qz, index.embeddings[Modality.IMAGE], image_items, top_k=2)
    print([(h.doc_id, round(h.score, 3)) for h in img_hits])
    assert len(img_hits) == 2

    print("\n=== score fusion with image bias ===")
    fused = multimodal_retrieve(
        enc,
        index,
        q,
        top_k_per_modality=2,
        query_bias={Modality.IMAGE: 1.5, Modality.TEXT: 1.0, Modality.AUDIO: 0.8},
        fused_top_k=5,
    )
    print([(h.doc_id, h.modality.value, round(h.score, 3)) for h in fused])
    assert len(fused) <= 5

    print("\n=== attention fusion ===")
    cand_z = []
    for h in fused[:3]:
        mod_items = index.items[h.modality]
        j = next(i for i, it in enumerate(mod_items) if it.doc_id == h.doc_id)
        cand_z.append(index.embeddings[h.modality][j])
    cand = torch.stack(cand_z, dim=0)
    attn = AttentionFusion(cfg.dim)
    w, fz = attn(qz, cand)
    print(
        f"attn_weights={[round(float(x.detach()), 3) for x in w]} "
        f"fused_dim={tuple(fz.shape)}"
    )
    assert fz.shape == (cfg.dim,)

    print("\n=== grounding ===")
    ctx = build_grounded_context(fused)
    print(ctx)
    answer = attach_claims_with_citations(
        ["The cat sits on the mat", "A bark is heard near the door"],
        fused,
    )
    print(answer)
    assert "[" in ctx

    print("SMOKE TEST OK")


smoke_test()
